# CREM: Plata (Silver) Consolidador de Datos
**Propósito**: Une todas las tablas de datos municipales del CREM (`raw.crem.edad_media_poblacion`, `raw.crem.indice_gini_y_distribucion_renta_p80_p20`, etc.) utilizando un `inner join` sobre el campo común `nombre`. Finalmente renombre esta columna a `municipio` y guarda el conjunto de datos completo en una tabla Delta de la capa `silver`.

In [1]:
import sys
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils

In [2]:
# Inicializar sesión de Spark usando las utilidades de TFM
spark = tfm_utils.get_spark_session(app_name="CREM_Silver_Full_Data")

In [3]:
# Lista de tablas raw a unir
tables = ["raw.crem.edad_media_poblacion", "raw.crem.indice_gini_y_distribucion_renta_p80_p20", "raw.crem.porcentaje_de_poblacion_de_65_o_mas",
            "raw.crem.porcentaje_poblacion_espanola", "raw.crem.porcentaje_de_poblacion_de_menos_de_18", "raw.crem.renta_neta_y_bruta_media_por_persona",
            "raw.crem.tamano_medio_hogar_y_pct_hogares_unipersonales", "raw.crem.tamano_poblacion", "raw.crem.transacciones_inmobiliarias"
        ]

# Cargar todas las tablas en un diccionario
dfs = {}
for t in tables:
    df_t = tfm_utils.read_from_delta(spark, t)
    dfs[t] = df_t

In [4]:
# Realizar los inner joins de forma secuencial por la columna 'nombre'
df_merged = dfs[tables[0]]
print(f"Base inicial ({tables[0]}): {df_merged.count()} filas")

for t in tables[1:]:
    df_merged = df_merged.join(dfs[t], on="nombre", how="inner")

Base inicial (raw.crem.edad_media_poblacion): 45 filas


In [5]:
# Renombrar la columna 'nombre' a 'municipio'
df_silver = df_merged.withColumnRenamed("nombre", "municipio")

In [6]:
# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos, ordenadas)
df_normalized = tfm_utils.normalize_df(df_silver)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_normalized)

,2015_distribucion_de_la_renta_p80_p20,2015_edad_media_poblacion,2015_indice_de_gini,2015_porcentaje_de_hogares_unipersonales,2015_porcentaje_de_poblacion_de_65_o_mas,2015_porcentaje_de_poblacion_de_menos_de_18,2015_porcentaje_poblacion_espanola,2015_renta_bruta_media_por_persona,2015_renta_neta_media_por_persona,2015_tamano_medio_del_hogar,...,iv_trimestre_2017_transacciones_inmobiliarias,iv_trimestre_2018_transacciones_inmobiliarias,iv_trimestre_2019_transacciones_inmobiliarias,iv_trimestre_2020_transacciones_inmobiliarias,iv_trimestre_2021_transacciones_inmobiliarias,iv_trimestre_2022_transacciones_inmobiliarias,iv_trimestre_2023_transacciones_inmobiliarias,iv_trimestre_2024_transacciones_inmobiliarias,iv_trimestre_2025_transacciones_inmobiliarias,municipio
0,2.9,38.4,34.4,26.0,14.6,22.0,75.3,8.737,7.654,2.9,...,158.0,189.0,145.0,157.0,237.0,251.0,262.0,351.0,309.0,san_pedro_del_pinatar
1,2.9,36.1,33.1,20.6,11.2,24.7,70.9,8.786,7.650,3.1,...,205.0,218.0,238.0,228.0,342.0,299.0,445.0,319.0,309.0,torre_pacheco
2,2.5,37.8,31.6,19.2,12.7,22.3,92.8,9.846,8.522,2.9,...,51.0,38.0,50.0,147.0,78.0,68.0,70.0,72.0,85.0,las_torres_de_cotillas
3,2.2,38.3,27.6,21.9,13.7,21.5,79.1,9.030,7.954,3.0,...,62.0,58.0,85.0,79.0,84.0,78.0,84.0,94.0,93.0,totana
4,2.5,45.6,30.4,31.8,26.3,15.5,94.1,10.169,9.061,2.6,...,5.0,2.0,1.0,1.0,7.0,3.0,7.0,3.0,4.0,ulea
5,2.9,36.8,33.6,21.6,12.2,23.7,90.4,8.679,7.631,3.0,...,50.0,60.0,66.0,74.0,69.0,96.0,78.0,130.0,114.0,la_union
6,3.0,38.0,31.9,29.2,13.2,21.2,94.9,9.536,8.437,2.5,...,18.0,22.0,9.0,27.0,53.0,20.0,18.0,31.0,32.0,villanueva_del_rio_segura
7,2.4,39.3,30.7,22.3,15.2,21.7,93.3,8.718,7.661,2.8,...,91.0,62.0,104.0,103.0,119.0,98.0,134.0,141.0,161.0,yecla
8,2.7,37.5,31.8,19.2,12.3,23.0,83.2,9.048,7.897,3.0,...,34.0,30.0,30.0,32.0,32.0,31.0,28.0,39.0,41.0,santomera
9,3.0,40.0,35.8,31.5,17.6,20.6,66.5,9.216,8.025,2.6,...,98.0,146.0,121.0,128.0,301.0,196.0,226.0,317.0,316.0,los_alcazares


In [7]:
# Ruta destino de la Delta Table en la capa Silver
silver_table_name = "silver.crem.crem_silver_full_data"
delta_path = tfm_utils.full_table_path(silver_table_name)

print(f"Escribiendo {df_normalized.count()} registros en la Delta Table: {silver_table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en la Delta Table: silver.crem.crem_silver_full_data
Destino: /tfm/delta_lake/silver/crem/crem_silver_full_data
¡Escritura en Delta Lake completada con éxito!
